# Семинар: работа со строками в Python — решения

Ноутбук основан на материалах лекции по форматированию строк, регулярным выражениям, сегментации текста, строкам в `numpy` и хешированию строк.

## Правила
1. В начале ноутбука задаётся `STUDENT_ID`
2. В большинстве заданий требуется не только написать код, но и сохранить позиции фрагментов, нормализовать данные, объяснить выбор правила или проверить крайние случаи.
3. В регулярных выражениях нужно использовать `fullmatch`, именованные группы, якоря, флаги или проверку через `datetime` там, где это указано.

## Структура
- Первая пара: задания 1–10.
- Вторая пара: задания 11–20.

In [ ]:
# Впишите свой идентификатор свой идентификатор.
STUDENT_ID = "123456"  # например: "123456" из "123456@edu.fa.ru"

In [ ]:
import re
import math
import time
import hashlib
import random
from datetime import datetime
from collections import defaultdict

try:
    import numpy as np
except ImportError:
    np = None

def variant_seed(student_id: str) -> int:
    return int.from_bytes(hashlib.sha256(student_id.encode("utf-8")).digest()[:4], "big")

def build_variant(student_id: str = STUDENT_ID) -> dict:
    rng = random.Random(variant_seed(student_id))
    surnames = ["Иванов", "Петрова", "Сидоров", "Ахметов", "Ким", "Галимова", "Орлова", "Макаров"]
    names = ["Иван", "Анна", "Павел", "Рустем", "Виктория", "Диана", "Олег", "Мария"]
    cities = ["Казань", "Уфа", "Самара", "Москва", "Екатеринбург", "Пермь"]
    goods = ["тетрадь", "ручка", "маркер", "карандаш", "папка", "стикеры", "линейка", "блокнот"]

    orders = []
    for _ in range(7):
        qty = rng.randint(1, 9)
        price = round(rng.uniform(35, 950), 2)
        discount = rng.choice([0, 0.03, 0.05, 0.07, 0.10, 0.15])
        orders.append({
            "order_id": rng.randint(10000, 99999),
            "buyer": f"{rng.choice(surnames)} {rng.choice(names)}",
            "city": rng.choice(cities),
            "item": rng.choice(goods),
            "qty": qty,
            "price": price,
            "discount": discount,
            "date": f"{rng.randint(1, 28):02d}.{rng.randint(1, 12):02d}.2026",
        })

    noisy_dates = [
        "29.02.2024", "29.02.2023", "31.04.2026", "30/04/2026",
        "01-13-2026", "7.5.2026", "07,05,2026", " 09.09.2026 ",
        f"{rng.randint(1, 28)}.{rng.randint(1, 12)}.2026",
        f"{rng.randint(29, 35)}.{rng.randint(1, 12)}.2026",
    ]

    file_paths = [
        rf"User\\Homework_{rng.choice(surnames)}.docs",
        rf"C:\\tmp\\course-work__{rng.choice(surnames).lower()}_v{rng.randint(1, 5)}.ipynb",
        f"/home/student/data/raw_text_{rng.randint(10,99)}.txt",
        f"report.final.{rng.randint(2023,2026)}.{rng.choice(['csv', 'xlsx', 'md'])}",
        f"архив/семинар 06/{rng.choice(surnames)}-{rng.choice(names)}.py",
    ]

    posts = [
        f"@{rng.choice(['corex','student_ai','ml_lab'])} Проверьте #Python и #RegExp в задаче {rng.randint(1,20)}: https://example.org/a/{rng.randint(100,999)}",
        f"Новый отчёт по #NLP, #токенизация и #строки. Автор: @{rng.choice(['ivan','anna','rustem'])}",
        f"Без ссылки, но с тегом #Дата{rng.randint(1,9)} и упоминанием @{rng.choice(['team','teacher'])}.",
        "Смешанный текст: https://aicorex.tech/demo?x=1, #Hashing, @qa_team!",
    ]

    logs = "\n".join([
        f"2026-04-24T12:{rng.randint(10,59):02d}:30.497Z INFO src.routes.recognition request_id={rng.randrange(16**8):08x} event=\"File saved successfully\"",
        f"2026-04-24T12:{rng.randint(10,59):02d}:31.100Z WARNING src.regex request_id={rng.randrange(16**8):08x} event=\"Ambiguous date token\"",
        f"2026-04-24T12:{rng.randint(10,59):02d}:32.777Z ERROR src.hash request_id={rng.randrange(16**8):08x} event=\"Duplicate candidate found\"",
        "broken line without required fields",
        f"2026-04-24T12:{rng.randint(10,59):02d}:33.050Z INFO src.pipeline request_id={rng.randrange(16**8):08x} event=\"Pipeline finished\"",
    ])

    html = (
        "<p>Перед <b>первым</b> блоком</p>"
        f"<a href=\"https://example.org/{rng.randint(10,99)}\">пример</a>"
        "<div><b>второй</b><span>не нужен</span><b>третий</b></div>"
    )

    name_rows = ["иванов, иван   ильич", " ПЕТРОВА АННА", "Сидоров П. С.", "ахметов, рустем ринатович", "ким виктория", "Орлова М.А."]
    rng.shuffle(name_rows)

    contacts = [
        "+7 (917) 123-45-67; ivan.petrov@example.ru; 92 15 123456",
        "8-999-000-11-22; bad_mail@@example; 1234 567890",
        "+7 843 222 33 44; student.ai@university.edu; 92 20 654321",
        "79991234567; name.surname@mail; 92 15 abcdef",
    ]

    multiline_notes = """TODO: переписать регулярное выражение
  todo: эта строка начинается с пробела и не должна попасть
FIXME: проверить сегментацию
текст TODO внутри строки не считается
fixme: нижний регистр тоже должен быть найден
"""

    ru_text = (
        "Проф. Иванов приехал в г. Казань 12.05.2026. "
        "Он сказал: «См. рис. 2.1 и табл. 3». "
        "Темп роста составил 3.14 процента. "
        "Студенты выполнили задание, т. е. получили зачёт! "
        "А что дальше? Новый этап начнётся завтра"
    )

    token_text = "Кружка-термос на 0.5л (50/64), цена — 1299.90 руб.; e-mail: test@example.ru"
    countries = ["USA", "Japan", "UK", "", "India", "China", "New Zealand", "Казахстан"]
    usernames = ["Barra_Fake", "Thamiris1996", "ReehMuruci", "ml_lab", "AI_Teacher", "student_01", "corex_systems", "nlp2026", "very_long_user_name_2026"]
    tweets = [
        "World Health Org official statement about #coronavirus and public health.",
        "I mean, Liberals are cheer-leading this #Coronavirus like it is a game.",
        "Новый материал по Python, строкам и регулярным выражениям #Python",
        "RT @ml_lab: регулярные выражения помогают искать даты, e-mail и токены",
        "Hashing strings is useful for duplicate detection and ids.",
        "Казань, Уфа и Самара обсуждают данные по обучению.",
    ] * 3
    rng.shuffle(tweets)
    hash_records = [
        " Иванов Иван: Python ", "иванов   иван: python", "Петрова Анна: NLP",
        "Петрова Анна: nlp ", "Сидоров Павел: regex", "сидоров павел: RegExp",
        f"Уникальная запись {rng.randint(100,999)}",
    ]

    return dict(
        seed=variant_seed(student_id), orders=orders, noisy_dates=noisy_dates, file_paths=file_paths,
        posts=posts, logs=logs, html=html, name_rows=name_rows, contacts=contacts,
        multiline_notes=multiline_notes, ru_text=ru_text, token_text=token_text,
        countries=countries, usernames=usernames, tweets=tweets, hash_records=hash_records,
    )

DATA = build_variant(STUDENT_ID)
print(f"Вариант: {STUDENT_ID}; seed={DATA['seed']}")
print("Пример заказа:", DATA["orders"][0])
print("Пример текста:", DATA["ru_text"][:120] + "...")

Вариант: 123456; seed=2375458543
Пример заказа: {'order_id': 18115, 'buyer': 'Иванов Мария', 'city': 'Екатеринбург', 'item': 'папка', 'qty': 6, 'price': 133.52, 'discount': 0.05, 'date': '08.10.2026'}
Пример текста: Проф. Иванов приехал в г. Казань 12.05.2026. Он сказал: «См. рис. 2.1 и табл. 3». Темп роста составил 3.14 процента. Сту...


## Задание 1. Табличный отчёт по заказам через f-строки

Напишите `format_order_table(orders) -> str`: таблица с колонками `order_id`, `buyer`, `city`, `item`, `qty`, `total`.
`total = qty * price * (1 - discount)`. Используйте f-строки, ширину/точность, `join`; все строки таблицы одинаковой длины.


In [ ]:
# ВАШ КОД

    ID | Покупатель             | Город          | Товар      | Кол-во |      Сумма
-----------------------------------------------------------------------------------
 18115 | Иванов Мария           | Екатеринбург   | папка      |      6 |     761.06
 46067 | Ахметов Иван           | Казань         | папка      |      6 |    5150.35
 35480 | Ахметов Виктория       | Уфа            | ручка      |      7 |    1816.19
 67682 | Петрова Диана          | Уфа            | карандаш   |      4 |     939.12
 53374 | Галимова Павел         | Москва         | стикеры    |      7 |    1215.22
 97853 | Галимова Олег          | Уфа            | карандаш   |      7 |    1524.53
 73841 | Макаров Рустем         | Казань         | блокнот    |      3 |     817.94


## Задание 2. Форматирование диагностической строки

Напишите `render_order_diagnostics(order) -> str`: дата, `order_id` в десятичном/hex/bin виде, скидка в процентах с 1 знаком, цена с разделителем тысяч и 2 знаками, итог с обязательным знаком.


In [ ]:
# ВАШ КОД

date=08.10.2026 | id=18115/0x46c3/100011011000011 | discount=5.0% | price=133.52 | total=+761.06
date=07.12.2026 | id=46067/0xb3f3/1011001111110011 | discount=5.0% | price=903.57 | total=+5150.35


## Задание 3. Склейка строк через `join`

Напишите `join_posts(posts, sep=' | ') -> str`: обрезать пробелы, игнорировать пустые строки, объединить через `sep.join(...)`. Дополнительно сравните время `join` и конкатенации в цикле на 5000 копиях постов.


In [ ]:
# ВАШ КОД

@ml_lab Проверьте #Python и #RegExp в задаче 7: https://example.org/a/719 | Новый отчёт по #NLP, #токенизация и #строки. Автор: @ivan | Без ссылки, но с тегом #Дата6 и упоминанием @team. | Смешанный текст: https://aicorex.tech/demo?x=1, #Hashing, @qa_team!
{'join_time_sec': 0.005804, 'concat_time_sec': 14.718247}


## Задание 4. Безопасная нормализация дат

Напишите `extract_valid_dates(values) -> list[str]`: принять даты с разделителями `.`, `/`, `-`, день/месяц 1-2 цифры, год 4 цифры; использовать `re.fullmatch` и `datetime`; вернуть корректные даты в ISO `YYYY-MM-DD`.


In [ ]:
# ВАШ КОД

['29.02.2024', '29.02.2023', '31.04.2026', '30/04/2026', '01-13-2026', '7.5.2026', '07,05,2026', ' 09.09.2026 ', '27.5.2026', '35.2.2026']
['2024-02-29', '2026-04-30', '2026-05-07', '2026-09-09', '2026-05-27']


## Задание 5. Разбор имён файлов и путей

Напишите `split_path_like_string(path) -> dict`: `parts` — непустые компоненты по `/`, `\`, `_`, пробелам; `extension` — расширение без точки; `has_version` — есть ли компонент `v2`, `v10` и т. п. Используйте `re.split`/`re.search`.


In [ ]:
# ВАШ КОД

User\\Homework_Макаров.docs -> {'parts': ['User', 'Homework', 'Макаров.docs'], 'extension': 'docs', 'has_version': False}
C:\\tmp\\course-work__орлова_v5.ipynb -> {'parts': ['C:', 'tmp', 'course-work', 'орлова', 'v5.ipynb'], 'extension': 'ipynb', 'has_version': True}
/home/student/data/raw_text_30.txt -> {'parts': ['home', 'student', 'data', 'raw', 'text', '30.txt'], 'extension': 'txt', 'has_version': False}
report.final.2026.xlsx -> {'parts': ['report.final.2026.xlsx'], 'extension': 'xlsx', 'has_version': False}
архив/семинар 06/Ким-Анна.py -> {'parts': ['архив', 'семинар', '06', 'Ким-Анна.py'], 'extension': 'py', 'has_version': False}


## Задание 6. Извлечение URL, хештегов и упоминаний с позициями

Напишите `extract_social_entities(posts) -> list[dict]`: найти URL, хештеги, упоминания; вернуть `post_index`, `type`, `value`, `start`, `end`. Хештеги привести к нижнему регистру, URL не должен включать финальные `, . !`.


In [ ]:
# ВАШ КОД

{'post_index': 0, 'type': 'mention', 'value': '@ml_lab', 'start': 0, 'end': 7}
{'post_index': 0, 'type': 'hashtag', 'value': '#python', 'start': 18, 'end': 25}
{'post_index': 0, 'type': 'hashtag', 'value': '#regexp', 'start': 28, 'end': 35}
{'post_index': 0, 'type': 'url', 'value': 'https://example.org/a/719', 'start': 48, 'end': 73}
{'post_index': 1, 'type': 'hashtag', 'value': '#nlp', 'start': 15, 'end': 19}
{'post_index': 1, 'type': 'hashtag', 'value': '#токенизация', 'start': 21, 'end': 33}
{'post_index': 1, 'type': 'hashtag', 'value': '#строки', 'start': 36, 'end': 43}
{'post_index': 1, 'type': 'mention', 'value': '@ivan', 'start': 52, 'end': 57}
{'post_index': 2, 'type': 'hashtag', 'value': '#дата6', 'start': 23, 'end': 29}
{'post_index': 2, 'type': 'mention', 'value': '@team', 'start': 44, 'end': 49}
{'post_index': 3, 'type': 'url', 'value': 'https://aicorex.tech/demo', 'start': 17, 'end': 42}
{'post_index': 3, 'type': 'hashtag', 'value': '#hashing', 'start': 48, 'end': 56}
{'po

## Задание 7. Парсер логов с именованными группами

Напишите `parse_logs(log_text) -> list[dict]`: извлечь корректные строки `timestamp LEVEL module request_id=... event="..."`; использовать `re.VERBOSE`, именованные группы; уровни `INFO|WARNING|ERROR`; timestamp преобразовать в `datetime`; некорректные строки пропустить.


In [ ]:
# ВАШ КОД

{'timestamp': datetime.datetime(2026, 4, 24, 12, 42, 30, 497000), 'level': 'INFO', 'module': 'src.routes.recognition', 'request_id': '231a5ac2', 'event': 'File saved successfully'}
{'timestamp': datetime.datetime(2026, 4, 24, 12, 49, 33, 50000), 'level': 'INFO', 'module': 'src.pipeline', 'request_id': 'b1f12d25', 'event': 'Pipeline finished'}
{'timestamp': datetime.datetime(2026, 4, 24, 12, 51, 32, 777000), 'level': 'ERROR', 'module': 'src.hash', 'request_id': '615165f0', 'event': 'Duplicate candidate found'}
{'timestamp': datetime.datetime(2026, 4, 24, 12, 52, 31, 100000), 'level': 'WARNING', 'module': 'src.regex', 'request_id': '5c1de8c1', 'event': 'Ambiguous date token'}


## Задание 8. Нежадное извлечение HTML-подобных тегов

Напишите `extract_html_fragments(html) -> dict`: вернуть `bold` — тексты всех `<b>...</b>`; `links` — список `{'href': ..., 'text': ...}` для `<a href="...">...</a>`. Использовать нежадные квантификаторы, HTML-парсер нельзя.


In [ ]:
# ВАШ КОД

{'bold': ['первым', 'второй', 'третий'], 'links': [{'href': 'https://example.org/54', 'text': 'пример'}]}


## Задание 9. Нормализация ФИО

Напишите `normalize_person_name(raw) -> str`: убрать лишние пробелы и запятые, привести слова к виду `Фамилия Имя Отчество`, инициалы разделить пробелом: `П. С.`, не `П.С.`.


In [ ]:
# ВАШ КОД

'Сидоров П. С.' -> Сидоров П. С.
'Орлова М.А.' -> Орлова М. А.
'ахметов, рустем ринатович' -> Ахметов Рустем Ринатович
' ПЕТРОВА АННА' -> Петрова Анна
'иванов, иван   ильич' -> Иванов Иван Ильич
'ким виктория' -> Ким Виктория


## Задание 10. Валидация контактных строк

Напишите `validate_contact_line(line) -> dict`: телефон нормализовать к `+7XXXXXXXXXX`, проверить e-mail, паспорт РФ `NN NN NNNNNN`. Вернуть `phone`, `email`, `passport`, `errors`.


In [ ]:
# ВАШ КОД

+7 (917) 123-45-67; ivan.petrov@example.ru; 92 15 123456
{'phone': '+79171234567', 'email': 'ivan.petrov@example.ru', 'passport': '92 15 123456', 'errors': []}
8-999-000-11-22; bad_mail@@example; 1234 567890
{'phone': '+79990001122', 'email': None, 'passport': None, 'errors': ['invalid_email', 'invalid_passport']}
+7 843 222 33 44; student.ai@university.edu; 92 20 654321
{'phone': '+78432223344', 'email': 'student.ai@university.edu', 'passport': '92 20 654321', 'errors': []}
79991234567; name.surname@mail; 92 15 abcdef
{'phone': '+79991234567', 'email': None, 'passport': None, 'errors': ['invalid_email', 'invalid_passport']}


## Задание 11. Поиск TODO/FIXME только в начале строк

Напишите `find_line_markers(text) -> list[dict]`: найти строки, которые начинаются с `TODO:` или `FIXME:` без начального пробела, регистр не важен. Используйте `re.MULTILINE` и `re.IGNORECASE`; вернуть `marker`, `text`, `start`, `end`.


In [ ]:
# ВАШ КОД

[{'marker': 'TODO', 'text': 'переписать регулярное выражение', 'start': 0, 'end': 37}, {'marker': 'FIXME', 'text': 'проверить сегментацию', 'start': 98, 'end': 126}, {'marker': 'FIXME', 'text': 'нижний регистр тоже должен быть найден', 'start': 165, 'end': 210}]


## Задание 12. Сегментация русскоязычного текста на основе правил

Напишите `split_sentences_ru(text) -> list[str]`: не делить после `г.`, `проф.`, `см.`, `рис.`, `табл.`, `т. е.`, `т. п.`, `т. д.`, не делить внутри `3.14`; делить по `.`, `!`, `?`; последнее предложение без знака тоже вернуть.


In [ ]:
# ВАШ КОД

'Проф. Иванов приехал в г. Казань 12.05.2026.'
'Он сказал: «См. рис. 2.1 и табл. 3».'
'Темп роста составил 3.14 процента.'
'Студенты выполнили задание, т. е. получили зачёт!'
'А что дальше?'
'Новый этап начнётся завтра'


## Задание 13. Токенизация с сохранением смещений

Напишите `tokenize_with_offsets(text) -> list[dict]`: выделить слова, числа с десятичной точкой, e-mail, знаки препинания и тире; вернуть `token`, `start`, `end`, `kind`; `text[start:end] == token`.


In [ ]:
# ВАШ КОД

{'token': 'Кружка-термос', 'start': 0, 'end': 13, 'kind': 'word'}
{'token': 'на', 'start': 14, 'end': 16, 'kind': 'word'}
{'token': '0.5', 'start': 17, 'end': 20, 'kind': 'number'}
{'token': 'л', 'start': 20, 'end': 21, 'kind': 'word'}
{'token': '(', 'start': 22, 'end': 23, 'kind': 'punct'}
{'token': '50', 'start': 23, 'end': 25, 'kind': 'number'}
{'token': '/', 'start': 25, 'end': 26, 'kind': 'punct'}
{'token': '64', 'start': 26, 'end': 28, 'kind': 'number'}
{'token': ')', 'start': 28, 'end': 29, 'kind': 'punct'}
{'token': ',', 'start': 29, 'end': 30, 'kind': 'punct'}
{'token': 'цена', 'start': 31, 'end': 35, 'kind': 'word'}
{'token': '—', 'start': 36, 'end': 37, 'kind': 'dash'}
{'token': '1299.90', 'start': 38, 'end': 45, 'kind': 'number'}
{'token': 'руб', 'start': 46, 'end': 49, 'kind': 'word'}
{'token': '.', 'start': 49, 'end': 50, 'kind': 'punct'}
{'token': ';', 'start': 50, 'end': 51, 'kind': 'punct'}
{'token': 'e-mail', 'start': 52, 'end': 58, 'kind': 'word'}
{'token': ':', 'sta

## Задание 14. Восстановление текста по токенам

Напишите `reconstruct_from_offsets(text, tokens) -> tuple[str, list]`: восстановить исходную строку по токенам и промежуткам между ними; вернуть восстановленную строку и список пропущенных промежутков.


In [ ]:
# ВАШ КОД

Кружка-термос на 0.5л (50/64), цена — 1299.90 руб.; e-mail: test@example.ru
[{'start': 13, 'end': 14, 'text': ' '}, {'start': 16, 'end': 17, 'text': ' '}, {'start': 21, 'end': 22, 'text': ' '}, {'start': 30, 'end': 31, 'text': ' '}, {'start': 35, 'end': 36, 'text': ' '}, {'start': 37, 'end': 38, 'text': ' '}, {'start': 45, 'end': 46, 'text': ' '}, {'start': 51, 'end': 52, 'text': ' '}, {'start': 59, 'end': 60, 'text': ' '}]


## Задание 15. Безопасный `numpy`-массив строк без обрезания

Напишите `make_safe_unicode_array(strings, min_len=0)`: создать `numpy`-массив `U{max_len}`, где `max_len=max(min_len, max длина строки)`, чтобы не обрезать строки. Проверьте замену пустой строки на `New Zealand`.


In [ ]:
# ВАШ КОД

['USA' 'Japan' 'UK' 'New Zealand' 'India' 'China' 'New Zealand'
 'Казахстан'] <U20


## Задание 16. Сравнение `object` и фиксированного Unicode dtype

Напишите `compare_string_array_memory(strings) -> dict`: создать `dtype=object` и `dtype=U{max_len}`, вернуть `nbytes`, `itemsize`, `avg_len`. Объясните, почему `object.nbytes` не включает память самих Python-строк.


In [ ]:
# ВАШ КОД

{'object_nbytes': 144, 'unicode_nbytes': 5256, 'object_itemsize': 8, 'unicode_itemsize': 292, 'avg_len': 64.5}
{'object_nbytes': 72, 'unicode_nbytes': 864, 'object_itemsize': 8, 'unicode_itemsize': 96, 'avg_len': 11.333}


## Задание 17. Векторизованные строковые операции в `numpy`

Напишите `count_contains_numpy(strings, needle) -> int`: привести строки и `needle` к нижнему регистру и посчитать строки, содержащие подстроку. Используйте `np.strings` при наличии, иначе `np.char`.


In [ ]:
# ВАШ КОД

python 3
regex 0
коронавирус 0
hash 3


## Задание 18. Оценка поиска подстроки

Напишите `benchmark_substring_search(strings, needle, repeat=200) -> dict`: сравнить Python-цикл `needle in s.lower()` и вариант из задания 17; вернуть время и одинаковость результатов. Не делать вывод «NumPy всегда быстрее».


In [ ]:
# ВАШ КОД

{'python_time_sec': 0.186968, 'numpy_time_sec': 0.421169, 'python_result': 150, 'numpy_result': 150, 'same_result': True}


## Задание 19. Стабильное хеширование нормализованных строк

Напишите `normalize_for_hash`, `stable_digest`, `group_duplicates_by_digest`. Используйте `hashlib.sha256`, а не `hash()`. Группировать только digest с более чем одной исходной записью; в группе хранить raw и normalized.


In [ ]:
# ВАШ КОД

{'c905582f6a360108': [{'raw': ' Иванов Иван: Python ', 'normalized': 'иванов иван: python'}, {'raw': 'иванов   иван: python', 'normalized': 'иванов иван: python'}], '1d0e3a1a40cbd4c3': [{'raw': 'Петрова Анна: NLP', 'normalized': 'петрова анна: nlp'}, {'raw': 'Петрова Анна: nlp ', 'normalized': 'петрова анна: nlp'}]}


## Задание 20. Эмпирическая оценка коллизий усечённого хеша

Напишите `simulate_hash_collisions(n=5000, bits=16, seed=42) -> dict`: сгенерировать строки, посчитать SHA-256, усечь до `bits`, вернуть число коллизий и приближение дня рождения `1-exp(-n*(n-1)/(2*2**bits))`. Сравните `bits=12,16,24`.


In [ ]:
# ВАШ КОД

12 {'n': 5000, 'bits': 12, 'unique': 2894, 'collisions': 2106, 'birthday_probability': 1.0}
16 {'n': 5000, 'bits': 16, 'unique': 4823, 'collisions': 177, 'birthday_probability': 1.0}
24 {'n': 5000, 'bits': 24, 'unique': 5000, 'collisions': 0, 'birthday_probability': 0.525223}
